In [ ]:
import yaml
import torch
from pathlib import Path
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

from qcm.models.hybrid_reupload import HybridReuploadClassifier, ReuploadClassifier
from qcm.components.reupload.reupload import QuantumHeadReupload
from qcm.data.datasets import get_dataloaders


In [2]:
def load_config(path: str) -> dict:
    with open(path, 'r') as f:
        return yaml.safe_load(f)
config = load_config('../configs/config_reupload.yaml')
dataset_type = config.get('dataset_type', 'pcam')

In [3]:
train_loader, val_loader = get_dataloaders(config)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ReuploadClassifier(config=config, use_quantum=True)
model.to(device)

ReuploadClassifier(
  (head): QuantumHeadReupload(
    (pad_layer): ZeroPad1d((0, 0))
  )
)

In [5]:
# Setup optimizer, scheduler, and loss function
optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=config['training']['lr'])
scheduler = CosineAnnealingLR(optimizer, T_max=config['training']['epochs'])

In [6]:
# Define run name and log file path
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"{dataset_type}_quantum_{timestamp}"
log_filepath = Path("./logs/training_log.csv")
log_filepath.parent.mkdir(exist_ok=True)

In [7]:
writer = SummaryWriter(log_dir=f"runs/test")
train_epoch_losses = []
val_epoch_losses = []

In [8]:
label = model.head.target_density_matrices

In [9]:
for l in label:
    # l = l.reshape(-1, 1)
    dm = l * torch.conj(l).T
    print(dm)

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])
tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1.]])


In [10]:
l.reshape(-1,1).ndim

2

In [11]:
for epoch in range(config['training']['epochs']):
    avg_train_loss = train_epoch(model, train_loader, optimizer, scheduler, epoch, config, writer)
    avg_val_loss = validate(model, val_loader, epoch, config, writer)
    
    train_epoch_losses.append(avg_train_loss)
    val_epoch_losses.append(avg_val_loss)

2025-10-06 16:57:00,570 - INFO - Starting training epoch 0
2025-10-06 16:57:00,570 - INFO - Epoch 0 START
2025-10-06 16:57:01,176 - INFO - Epoch 0 END: Loss=0.2010, F1 Score=0.6234
2025-10-06 16:57:01,176 - INFO - Finished training epoch 0. Avg Loss: 0.2970, Accuracy: 0.6234
2025-10-06 16:57:01,176 - INFO - Starting validation for epoch 0
2025-10-06 16:57:01,203 - INFO - Validation Epoch 0: Loss=0.4863, F1 Score=0.7784
2025-10-06 16:57:01,204 - INFO - Starting training epoch 1
2025-10-06 16:57:01,204 - INFO - Epoch 1 START
2025-10-06 16:57:01,720 - INFO - Epoch 1 END: Loss=0.1702, F1 Score=0.8045
2025-10-06 16:57:01,720 - INFO - Finished training epoch 1. Avg Loss: 0.1901, Accuracy: 0.8045
2025-10-06 16:57:01,721 - INFO - Starting validation for epoch 1
2025-10-06 16:57:01,747 - INFO - Validation Epoch 1: Loss=0.4769, F1 Score=0.7784
2025-10-06 16:57:01,748 - INFO - Starting training epoch 2
2025-10-06 16:57:01,748 - INFO - Epoch 2 START
2025-10-06 16:57:02,267 - INFO - Epoch 2 END: Lo

In [39]:
import pennylane as qml
inp = train_loader.dataset[100][0]
label = train_loader.dataset[100][1]
dm_label = model.head.target_density_matrices[label.int()].squeeze()
params = model.head.q_params
print(qml.draw(model.head.circuit)(inp, params, dm_label))

0: ──Rot(4.65,4.79,4.89)──Rot(-0.20,1.06,0.12)──╭●────╭X──Rot(4.65,4.79,4.89) ···
1: ──Rot(4.79,4.57,4.83)──Rot(-0.28,2.74,1.25)──╰X─╭●─│───Rot(4.79,4.57,4.83) ···
2: ──Rot(4.82,4.81,4.86)──Rot(-1.53,-1.57,0.39)────╰X─╰●──Rot(4.82,4.81,4.86) ···

0: ··· ──Rot(0.63,0.05,1.67)───╭●────╭X──Rot(4.65,4.79,4.89)──Rot(1.45,-0.78,0.92)───╭●────╭X─┤ ···
1: ··· ──Rot(1.32,-1.27,0.23)──╰X─╭●─│───Rot(4.79,4.57,4.83)──Rot(-0.05,-1.39,-0.15)─╰X─╭●─│──┤ ···
2: ··· ──Rot(-2.59,-0.97,1.25)────╰X─╰●──Rot(4.82,4.81,4.86)──Rot(-0.08,-0.15,1.12)─────╰X─╰●─┤ ···

0: ···  ╭<𝓗(M0)>
1: ···  ├<𝓗(M0)>
2: ···  ╰<𝓗(M0)>

M0 = 
tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1.]])


In [13]:
x = torch.tensor(train_loader.dataset[:][0]).repeat_interleave(2, 0)

In [14]:
train_loader.dataset[:][0].shape

(1024, 9)

In [15]:
x.shape

torch.Size([2048, 9])

In [16]:
label = train_loader.dataset[:][1]

In [17]:
y = model.head.target_labels.reshape(1, -1)
y = torch.repeat_interleave(y, 1024, dim=0).reshape(-1, 1)

In [18]:
y.shape

torch.Size([2048, 1])

In [19]:
_, pred = model(x,y).reshape(-1,2).min(dim=1)

In [20]:
from torchmetrics.classification import BinaryF1Score
metric = BinaryF1Score()
metric.update(pred, label)
metric_val = metric.compute()
metric_val.item()

0.8170594573020935

In [21]:
pred, label

(tensor([0, 1, 1,  ..., 0, 0, 0]), tensor([0., 1., 1.,  ..., 1., 0., 0.]))

In [22]:
xval = torch.tensor(val_loader.dataset[:10][0]).repeat_interleave(2, 0)

In [23]:
val_loader.dataset[:10][1]

tensor([1., 1., 1., 1., 1., 1., 0., 1., 1., 1.])

In [24]:
y = model.head.target_labels.reshape(1, -1)
y = torch.repeat_interleave(y, 10, dim=0).reshape(-1, 1)

In [25]:
model(xval,y).reshape(-1,2).min(dim=1)

torch.return_types.min(
values=tensor([0.5816, 0.5131, 0.4424, 0.4820, 0.4565, 0.1816, 0.3759, 0.4610, 0.4681,
        0.4033], dtype=torch.float64, grad_fn=<MinBackward0>),
indices=tensor([1, 0, 1, 0, 0, 1, 0, 0, 0, 1]))

In [26]:
model.head.num_qubits

3

In [27]:
x = torch.zeros(10)

In [28]:
idx = list(range(10))

In [29]:
idx

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [30]:
x[0] = 1

In [31]:
x

tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [32]:
available_idx = list(range(10))

In [33]:
available_idx

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [34]:
available_idx.remove(0)
# available_idx.remove(9)

In [35]:
available_idx

[1, 2, 3, 4, 5, 6, 7, 8, 9]

In [36]:
used_index = [0]

In [37]:
torch.abs(torch.tensor(available_idx).reshape(-1,1)- torch.tensor(used_index)).prod(1)

tensor([1, 2, 3, 4, 5, 6, 7, 8, 9])

In [38]:
k.argmax().item()

NameError: name 'k' is not defined

In [ ]:
num_classes = 8
num_qubits = 4

def get_target_states() -> torch.Tensor:
    """Computes the target states associtated with the different labels

    Raises:
        ValueError: _description_

    Returns:
        torch.Tensor: target states
    """

    
    def get_ideal_state(used_index, available_index) -> int:
        us_idx = torch.tensor(used_index)
        av_idx = torch.tensor(available_index).reshape(-1, 1)
        prod_distance = torch.abs(av_idx - us_idx).prod(1)
        return prod_distance.argmax().item()


    target_states = torch.zeros(num_classes, 2**num_qubits)
    available_index = list(range(2**num_qubits))

    used_index = []
    for idx in range(num_classes):
        if idx == 0:
            state_idx = 0
        else:
            state_idx = available_index[get_ideal_state(used_index, available_index)]

        target_states[idx, state_idx] = 1
        used_index.append(state_idx)
        available_index.remove(state_idx)
    return target_states

get_target_states().sum(0)

tensor([1., 0., 1., 0., 1., 0., 0., 1., 0., 0., 1., 0., 1., 0., 1., 1.])

In [ ]:
available_idx

[1, 2, 3, 4, 5, 6, 7, 8, 9]